# Kaggle: tái lập gần đúng đánh giá tutor LLM trên MRBench

Notebook này tải trọng số từ Hugging Face và chạy inference cục bộ trên GPU Kaggle (4-bit), không dùng API suy luận trả phí.

- Các họ model tutor mã nguồn mở trong paper được đại diện bằng Llama 3.1 8B Instruct, Phi-3 Mini Instruct và Mistral 7B Instruct. Paper không công bố checkpoint/revision đầy đủ, nên các ID này là lựa chọn gần đúng; Mistral v0.2 cũng cần được xem là giả định.
- Model thêm để so sánh: Qwen2.5-7B-Instruct.
- Llama 3.1 405B được paper dùng nhưng không phù hợp để nạp cục bộ trên GPU Kaggle thông thường.
- Paper dùng cả Prometheus2 và Llama-3.1-8B làm critic; notebook hiện chỉ chạy Prometheus2 trên mẫu nhỏ của các response mới. Đây chưa phải tái lập thí nghiệm Annotation Correlation (AC) của paper.

Paper không công bố đầy đủ Hugging Face IDs/revisions, decoding settings hay phần cứng. Llama 3.1 yêu cầu chấp nhận điều khoản Meta trên Hugging Face và thêm HF_TOKEN vào Kaggle Secrets.

**Đây là tái lập gần đúng, không phải tái lập từng bit hoặc tái lập đầy đủ mọi thí nghiệm.** File MRBench V1 hiện có phân bố khác số liệu paper; dùng tokenizer chat template của từng model và lượng tử hóa 4-bit cũng có thể đổi output. Notebook ghi lại revision và tham số đã chạy.

Paper đánh giá DAMR bằng nhãn người gán. Response mới chưa có nhãn như vậy; notebook xuất CSV để người gán nhãn và giữ riêng điểm Prometheus2 (proxy), không trình bày điểm judge như human DAMR.


## Quy trình

1. Audit file MRBench V1 và tính lại DAMR của response đã có nhãn; so với Table 3 của paper.
2. Sinh mới response từ cùng hội thoại bằng prompt Figure 2 trong paper, chạy greedy decoding cố định.
3. Xuất template CSV gán nhãn response mới. Chỉ so sánh human DAMR sau khi gán nhãn theo guideline paper.
4. Tùy chọn dùng Prometheus2 chấm một sample theo rubric paper; báo cáo riêng vì judge có thể không đồng thuận với con người.

F1 của NB/RF/BERT đã chạy trước đó là metric cho classifier dự đoán annotation; không so sánh trực tiếp với DAMR đánh giá tutor.

## Kaggle setup

- Bật Accelerator: GPU (T4 ×2 nếu được cấp; kiểm tra tên GPU thực tế), Internet: On.
- Thêm Kaggle Secret tên HF_TOKEN. Llama yêu cầu trước đó chấp nhận license tại Hugging Face.
- Đặt MAX_DIALOGUES = 12 để chạy pilot; đặt None để chạy toàn bộ release.
- Kết quả nằm dưới /kaggle/working/mrbench_open_llm. Model 405B cần endpoint/server nhiều GPU riêng.
- Chạy đầy đủ mặc định sẽ sinh 4 × 192 = 768 response tutor, rồi Prometheus2 chấm 4 × 12 × 8 = 384 tiêu chí. Ước lượng thực tế phụ thuộc GPU, tốc độ tải model và kết quả pilot; xem phần ước lượng trong báo cáo đi kèm.


## Ước lượng thời gian (chưa đo trên phiên Kaggle thực)

- Full run mặc định: 768 response tutor + 384 lượt judge theo từng chiều. Trên GPU Kaggle T4 ×2, ước tính khoảng 45–100 phút nếu checkpoint đã nằm trong cache; lần chạy đầu khoảng 1–3 giờ do cài thư viện và tải khoảng 60–70 GB trọng số. Tốc độ mạng, tình trạng cấp GPU, model access và response dài/ngắn có thể làm thay đổi đáng kể.
- Pilot 12 hội thoại nhưng giữ đủ 4 tutor model và judge vẫn tải đủ các checkpoint; ước tính khoảng 40–120 phút lần đầu. Để kiểm tra đường chạy trước, chọn 1 tutor trong RUN_MODELS, đặt MAX_DIALOGUES = 12 và RUN_PROMETHEUS_JUDGE = False.
- Con số trên là ước lượng kỹ thuật, không phải benchmark thực đo. Generation CSV có generation_seconds để thay bằng số đo sau pilot.
- Nếu triển khai AC đầy đủ cho paper, riêng hai critic × 1,589 response × 8 chiều = 25,424 lượt judge; chạy tuần tự có thể mất khoảng 7–25 giờ và có nguy cơ vượt giới hạn phiên Kaggle 12 giờ. Notebook hiện chưa triển khai chế độ này.

Trọng số được tải về cache tạm `/kaggle/temp/hf_cache`; lượng tử hóa 4-bit giảm VRAM lúc inference nhưng không đồng nghĩa chỉ tải file 4-bit.


In [ ]:
%pip -q install "transformers==4.46.3" "accelerate>=0.34.2" "bitsandbytes>=0.44.1" sentencepiece huggingface_hub

In [ ]:
import os, re, json, time, random, warnings, hashlib
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import transformers
from IPython.display import display

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("Bật GPU trong Kaggle Notebook Settings trước khi nạp model 7B/8B.")

gpu_name = torch.cuda.get_device_name(0)
gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print({"GPU": gpu_name, "VRAM_GB": round(gpu_memory_gb, 1),
       "PyTorch": torch.__version__, "Transformers": transformers.__version__,
       "CUDA": torch.version.cuda})
assert gpu_memory_gb >= 10, "VRAM dưới 10 GB; model 7B/8B 4-bit có thể không chạy ổn định."

In [ ]:
from huggingface_hub import login
import requests

HF_TOKEN = os.environ.get("HF_TOKEN", "")
try:
    from kaggle_secrets import UserSecretsClient
    if not HF_TOKEN:
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass

if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("Đã đăng nhập Hugging Face.")
else:
    print("Không có HF_TOKEN. Model công khai vẫn tải được; Llama gated có thể bị bỏ qua.")

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/kaushal0494/UnifyingAITutorEvaluation/main/MRBench/MRBench_V1.json"
EXPECTED_DATA_SHA256 = "77a33ab81ca9c773d22faee2be7a7bb43b203a06497a5daa35d715f9ada3a6f3"  # snapshot in this workspace
OUT_DIR = Path("/kaggle/working/mrbench_open_llm")
CACHE_DIR = Path("/kaggle/temp/hf_cache")
OUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

candidate_paths = [
    Path("/kaggle/input/mrbench-v1/MRBench_V1.json"),
    Path("/kaggle/input/MRBench_V1.json"),
]
if Path("/kaggle/input").exists():
    candidate_paths += list(Path("/kaggle/input").rglob("MRBench_V1.json"))
DATA_PATH = next((p for p in candidate_paths if p.exists()), None)

if DATA_PATH is None:
    response = requests.get(DATA_URL, timeout=120)
    response.raise_for_status()
    DATA_PATH = OUT_DIR / "MRBench_V1.json"
    DATA_PATH.write_bytes(response.content)

dialogues = json.loads(DATA_PATH.read_text(encoding="utf-8"))
assert isinstance(dialogues, list)
source_counts = Counter(str(d.get("Data", "Unknown")) for d in dialogues)
data_sha256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
if data_sha256.lower() != EXPECTED_DATA_SHA256:
    raise RuntimeError(f"Unexpected MRBench file hash: {data_sha256}; expected {EXPECTED_DATA_SHA256}. Check the release snapshot before comparing results.")
print("Data:", DATA_PATH)
print("SHA-256:", data_sha256)
print("Dialogue objects:", len(dialogues))
print("Sources:", dict(source_counts))
print("Unique conversation IDs:", len({str(d.get("conversation_id")) for d in dialogues}))
print("Paper reports 192 dialogues: 60 Bridge, 132 MathDial, 1,596 responses.")
print("Giữ nguyên số liệu thực tế của release; không âm thầm sửa khác biệt.")

## Tái tính DAMR cho response đã phát hành

Đây là phân tích trên response/annotation có sẵn, không gọi model. DAMR là tỷ lệ response nhận nhãn mong muốn: Yes cho nhận diện lỗi, vị trí lỗi, guidance, actionability, coherence, humanlikeness; No cho revealing answer; Encouraging cho tone.

In [ ]:
DIMENSIONS = [
    "Mistake_Identification", "Mistake_Location", "Revealing_of_the_Answer",
    "Providing_Guidance", "Actionability", "Coherence", "Tutor_Tone", "Humanlikeness",
]
DESIRED = {
    "Mistake_Identification": "Yes", "Mistake_Location": "Yes",
    "Revealing_of_the_Answer": "No", "Providing_Guidance": "Yes",
    "Actionability": "Yes", "Coherence": "Yes", "Tutor_Tone": "Encouraging",
    "Humanlikeness": "Yes",
}
PAPER_TABLE3 = {
    "Novice": [43.33,16.67,80.00,11.67,1.67,50.00,90.00,35.00],
    "Expert": [76.04,63.02,90.62,67.19,76.04,79.17,92.19,87.50],
    "Llama-3.1-8B": [80.21,54.69,73.96,45.31,42.71,80.73,19.79,93.75],
    "Phi3": [28.65,26.04,73.96,17.71,11.98,39.58,45.31,52.08],
    "Gemini": [63.02,39.58,67.71,37.50,42.71,56.77,21.88,68.23],
    "Sonnet": [85.42,69.79,94.79,59.38,60.94,88.54,54.69,96.35],
    "Mistral": [93.23,73.44,86.46,63.54,70.31,86.98,15.10,95.31],
    "GPT-4": [94.27,84.38,53.12,76.04,46.35,90.17,37.50,89.62],
    "Llama-3.1-405B": [94.27,84.38,80.73,77.08,74.48,91.67,16.15,90.62],
}
TUTOR_KEYS = {
    "Gemini": "Gemini", "Phi3": "Phi3", "Llama318B": "Llama-3.1-8B",
    "Llama31405B": "Llama-3.1-405B", "Mistral": "Mistral", "Expert": "Expert",
    "GPT4": "GPT-4", "Sonnet": "Sonnet", "Novice": "Novice",
}

def canonical_dimension(raw_key):
    key = re.sub(r"[^a-z]", "", str(raw_key).casefold())
    return {
        "mistakeidentification": "Mistake_Identification",
        "mistakelocation": "Mistake_Location",
        "revealingoftheanswer": "Revealing_of_the_Answer",
        "providingguidance": "Providing_Guidance",
        "actionability": "Actionability",
        "coherence": "Coherence",
        "tutortone": "Tutor_Tone",
        "humanlikeness": "Humanlikeness",
        "humanness": "Humanlikeness",
    }.get(key)

def normalize_label(dimension, value):
    if value is None:
        return None
    text = re.sub(r"\s+", " ", str(value).strip().casefold())
    if dimension == "Revealing_of_the_Answer":
        if text in {"no", "no answer", "does not reveal"}:
            return "No"
        if text.startswith("yes"):
            if "incorrect" in text or "wrong" in text:
                return "Yes (but the answer is incorrect)"
            if "correct" in text or text == "yes":
                return "Yes (and the answer is correct)"
        return None
    if dimension == "Tutor_Tone":
        return {"encouraging":"Encouraging", "neutral":"Neutral", "offensive":"Offensive"}.get(text)
    return {"yes":"Yes", "to some extent":"To some extent", "no":"No"}.get(text)

release_rows = []
for dialogue_idx, dialogue in enumerate(dialogues):
    for raw_tutor, obj in (dialogue.get("anno_llm_responses") or {}).items():
        normalized = {}
        for raw_dim, raw_value in (obj.get("annotation") or {}).items():
            dim = canonical_dimension(raw_dim)
            if dim:
                normalized[dim] = normalize_label(dim, raw_value)
        release_rows.append({
            "dialogue_index": dialogue_idx,
            "conversation_id": str(dialogue.get("conversation_id", "")),
            "source": str(dialogue.get("Data", "Unknown")),
            "tutor": TUTOR_KEYS.get(raw_tutor, raw_tutor),
            "response": str(obj.get("response", "")),
            **{d: normalized.get(d) for d in DIMENSIONS},
        })
release_df = pd.DataFrame(release_rows)
print("Response rows:", len(release_df))
print("Rows by source:", release_df["source"].value_counts().to_dict())
print("Rows by tutor:", release_df.groupby("tutor")["response"].count().to_dict())
print("Complete labels:", release_df[DIMENSIONS].notna().sum().to_dict())

In [ ]:
release_rows = []
for tutor, group in release_df.groupby("tutor", sort=False):
    for dim in DIMENSIONS:
        labels = group[dim].dropna()
        release_rows.append({
            "tutor": tutor, "dimension": dim,
            "release_DAMR_pct": 100 * (labels == DESIRED[dim]).mean() if len(labels) else np.nan,
            "n_release": int(len(labels)),
        })
release_damr = pd.DataFrame(release_rows)
paper_damr = pd.DataFrame([
    {"tutor": tutor, "dimension": dim, "paper_DAMR_pct": score}
    for tutor, scores in PAPER_TABLE3.items()
    for dim, score in zip(DIMENSIONS, scores)
])
release_vs_paper = paper_damr.merge(release_damr, on=["tutor","dimension"], how="left")
release_vs_paper["release_minus_paper_pp"] = (
    release_vs_paper["release_DAMR_pct"] - release_vs_paper["paper_DAMR_pct"]
)
display(release_vs_paper.round(2))
release_vs_paper.to_csv(OUT_DIR / "paper_vs_current_release_damr.csv", index=False)
print("Saved", OUT_DIR / "paper_vs_current_release_damr.csv")

## Model IDs và cách chạy

Paper nêu Llama-3.1-8B, Llama-3.1-405B, Phi3 và Mistral nhưng không công bố đầy đủ Hugging Face checkpoint ID/revision. Notebook dùng các bản Instruct gần tương ứng; Mistral-7B-Instruct-v0.2 là giả định có ghi nhận. Model 405B được ghi nhận nhưng bỏ khỏi chạy local vì không phù hợp VRAM Kaggle.


In [ ]:
MODEL_REGISTRY = {
    "llama31_8b": {
        "model_id": "meta-llama/Llama-3.1-8B-Instruct",
        "trust_remote_code": False,
        "paper_name": "Llama-3.1-8B",
    },
    "phi3_mini": {
        "model_id": "microsoft/Phi-3-mini-4k-instruct",
        "trust_remote_code": True,
        "paper_name": "Phi3",
    },
    "mistral_7b": {
        "model_id": "mistralai/Mistral-7B-Instruct-v0.2",
        "trust_remote_code": False,
        "paper_name": "Mistral (checkpoint gần đúng)",
    },
    "qwen25_7b": {
        "model_id": "Qwen/Qwen2.5-7B-Instruct",
        "trust_remote_code": False,
        "paper_name": "Added model",
    },
}
RUN_MODELS = ["llama31_8b", "phi3_mini", "mistral_7b", "qwen25_7b"]
MAX_DIALOGUES = None   # None = all release rows; set 12 for pilot.
LOAD_IN_4BIT = True
MAX_NEW_TOKENS = 128
GENERATION_FILE = OUT_DIR / "regenerated_tutor_responses.csv"
display(pd.DataFrame([{"alias": k, **v} for k,v in MODEL_REGISTRY.items()]))

In [ ]:
def build_messages(dialogue):
    source = str(dialogue.get("Data", ""))
    history = str(dialogue.get("conversation_history", "")).strip()
    if source == "Bridge":
        system = (
            "You are an experienced elementary school math teacher, and you are going "
            "to respond to a student's mistake in a useful and caring way."
        )
        topic = str(dialogue.get("Topic", "")).strip()
        user = (
            f"The problem your student is solving is on the topic: {topic}\n"
            f"Conversation History: {history}\n"
            "Tutor response (maximum one sentence that is most appropriate given "
            "topic and conversation history):"
        )
    else:
        system = (
            "You are an experienced middle school math teacher, and you are going "
            "to respond to a student's mistake in a useful and caring way."
        )
        user = (
            "The conversation history of the problem your student is solving is: "
            f"{history}\n"
            "Tutor response (maximum one sentence that aligns most appropriately "
            "with conversation history):"
        )
    return [{"role":"system","content":system},{"role":"user","content":user}]

def load_local_model(model_id, trust_remote_code=False):
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    tokenizer = AutoTokenizer.from_pretrained(
        model_id, token=HF_TOKEN or None, trust_remote_code=trust_remote_code,
        cache_dir=str(CACHE_DIR),
    )
    quant = None
    if LOAD_IN_4BIT:
        quant = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16,
        )
    model = AutoModelForCausalLM.from_pretrained(
        model_id, token=HF_TOKEN or None,
        trust_remote_code=trust_remote_code,
        torch_dtype=torch.float16, quantization_config=quant,
        device_map="auto", low_cpu_mem_usage=True, cache_dir=str(CACHE_DIR),
    )
    model.eval()
    return tokenizer, model

def render_chat_prompt(tokenizer, messages):
    if getattr(tokenizer, "chat_template", None):
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return (
        f"<|system|>\n{messages[0]['content']}<|end|>\n"
        f"<|user|>\n{messages[1]['content']}<|end|>\n<|assistant|>\n"
    )

def make_dialogue_frame():
    records = [{
        "dialogue_index": i,
        "conversation_id": str(d.get("conversation_id", "")),
        "source": str(d.get("Data", "Unknown")),
        "topic": str(d.get("Topic", "")),
        "history": str(d.get("conversation_history", "")),
        "raw_dialogue": d,
    } for i,d in enumerate(dialogues)]
    frame = pd.DataFrame(records)
    if MAX_DIALOGUES is None or MAX_DIALOGUES >= len(frame):
        return frame
    chunks = []
    for _, group in frame.groupby("source", sort=False):
        n = max(1, round(MAX_DIALOGUES * len(group) / len(frame)))
        chunks.append(group.sample(n=min(n,len(group)), random_state=SEED))
    sample = pd.concat(chunks).drop_duplicates("dialogue_index")
    if len(sample) < MAX_DIALOGUES:
        rest = frame.loc[~frame.dialogue_index.isin(sample.dialogue_index)]
        sample = pd.concat([sample, rest.sample(
            n=min(MAX_DIALOGUES-len(sample),len(rest)), random_state=SEED
        )])
    return sample.sort_values("dialogue_index").head(MAX_DIALOGUES).reset_index(drop=True)

def generate_for_model(alias):
    config = MODEL_REGISTRY[alias]
    print("Loading", alias, config["model_id"])
    tokenizer, model = load_local_model(config["model_id"], config["trust_remote_code"])
    revision = getattr(model.config, "_commit_hash", None)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    existing = pd.read_csv(GENERATION_FILE) if GENERATION_FILE.exists() else pd.DataFrame()
    done = set()
    if not existing.empty:
        old = existing[existing.model_alias.astype(str) == alias]
        if not old.empty:
            signature_columns = ["model_id", "model_revision", "quantization", "seed", "max_new_tokens"]
            old_signatures = old[signature_columns].fillna("").astype(str).drop_duplicates()
            current_signature = tuple(str(x) for x in [
                config["model_id"], revision,
                "4bit_nf4" if LOAD_IN_4BIT else "fp16", SEED, MAX_NEW_TOKENS,
            ])
            if any(tuple(row) != current_signature for row in old_signatures.to_numpy()):
                raise RuntimeError(
                    "Generation file contains a different model revision/config for this alias. "
                    "Move/delete regenerated_tutor_responses.csv before starting a new run."
                )
            done = set(zip(old.model_alias.astype(str), old.dialogue_index.astype(int)))

    frame = make_dialogue_frame()
    new_count = 0
    for row in frame.itertuples(index=False):
        key = (alias, int(row.dialogue_index))
        if key in done:
            continue
        prompt = render_chat_prompt(tokenizer, build_messages(row.raw_dialogue))
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=3072)
        device = model.get_input_embeddings().weight.device
        inputs = {k:v.to(device) for k,v in inputs.items()}
        started = time.time()
        with torch.inference_mode():
            output = model.generate(
                **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, num_beams=1,
                pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id,
                use_cache=True,
            )
        new_tokens = output[0, inputs["input_ids"].shape[1]:]
        answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        record = {
            "dialogue_index": int(row.dialogue_index),
            "conversation_id": row.conversation_id, "source": row.source,
            "topic": row.topic, "history": row.history,
            "model_alias": alias, "model_id": config["model_id"],
            "model_revision": revision,
            "quantization": "4bit_nf4" if LOAD_IN_4BIT else "fp16",
            "seed": SEED, "do_sample": False,
            "max_new_tokens": MAX_NEW_TOKENS, "response": answer,
            "generation_seconds": round(time.time()-started, 3),
        }
        pd.DataFrame([record]).to_csv(
            GENERATION_FILE, mode="a", header=not GENERATION_FILE.exists(),
            index=False, encoding="utf-8",
        )
        done.add(key)
        new_count += 1
        if new_count % 10 == 0:
            print(f"{alias}: {new_count}/{len(frame)} newly generated")

    del model, tokenizer
    torch.cuda.empty_cache()
    print("Finished", alias, "revision", revision)

for alias in RUN_MODELS:
    try:
        generate_for_model(alias)
    except Exception as exc:
        print(f"MODEL FAILED: {alias}: {type(exc).__name__}: {exc}")
        print("Kiểm tra quyền truy cập HF, Internet, phiên bản model và VRAM; model khác vẫn sẽ chạy.")

if GENERATION_FILE.exists():
    generations = pd.read_csv(GENERATION_FILE)
    display(generations.groupby(["model_alias","source"]).agg(
        n=("response","size"),
        avg_response_chars=("response",lambda x:x.fillna("").str.len().mean()),
        avg_seconds=("generation_seconds","mean"),
    ).round(2))
    display(generations.head(3))
else:
    generations = pd.DataFrame()
    print("Chưa tạo được response nào.")

## Xuất template gán nhãn người

Nhãn người là điều kiện để so sánh lại với DAMR trong paper. Dùng guideline annotation của repository và giữ nguyên các lựa chọn nhãn.

In [ ]:
if not GENERATION_FILE.exists():
    generations = pd.DataFrame()
else:
    generations = pd.read_csv(GENERATION_FILE)

if not generations.empty:
    annotation_template = generations[
        ["dialogue_index","conversation_id","source","topic","history",
         "model_alias","model_id","response"]
    ].copy()
    for dim in DIMENSIONS:
        annotation_template[dim] = ""
    template_path = OUT_DIR / "human_annotation_template.csv"
    annotation_template.to_csv(template_path, index=False, encoding="utf-8-sig")
    print("Template:", template_path)
    display(annotation_template.head())
else:
    print("Run a tutor model first.")

## Tùy chọn: Prometheus2 chấm mẫu theo rubric paper

Paper dùng Prometheus2 và Llama-3.1-8B làm critic, sau đó tính Pearson AC giữa nhãn critic và nhãn người trên responses của các tutor trong MRBench. Notebook hiện chỉ thực hiện một phần nhỏ: Prometheus2 chấm tối đa 12 response mới cho mỗi model tutor, riêng từng dimension. Vì vậy kết quả ở đây là judge-DAMR tham khảo; không phải các Table 5–6 (AC) của paper.

Để tái lập AC đầy đủ cần chạy cả hai critic trên các response gốc đã gán nhãn (tối đa 1,589 response × 8 chiều × 2 critic = 25,424 lượt chấm theo release hiện tại), rồi tính Pearson correlation trên từng nhóm tutor/chiều. Chi phí thời gian sẽ lớn hơn nhiều so với sample hiện tại.


In [ ]:
JUDGE_MODEL_ID = "prometheus-eval/prometheus-7b-v2.0"
JUDGE_PER_TUTOR_N = 12
RUN_PROMETHEUS_JUDGE = True
JUDGE_FILE = OUT_DIR / "prometheus2_judgements.csv"

DEFINITIONS = {
    "Mistake_Identification": "Has the tutor identified a mistake in a student's response?",
    "Mistake_Location": "Does the tutor's response accurately point to a genuine mistake and its location?",
    "Revealing_of_the_Answer": "Does the tutor reveal the final answer (whether correct or not)?",
    "Providing_Guidance": "Does the tutor offer correct and relevant guidance, such as an explanation, elaboration, hint, examples, and so on?",
    "Coherence": "Is the tutor's response logically consistent with the student's previous response?",
    "Actionability": "Is it clear from the tutor's feedback what the student should do next?",
    "Tutor_Tone": "Is the tutor's response encouraging, neutral, or offensive?",
    "Humanlikeness": "Does the tutor's response sound natural, rather than robotic or artificial?",
}
RUBRICS = {
    "Mistake_Identification": "Score 1: Yes. Score 2: To some extent. Score 3: No.",
    "Mistake_Location": "Score 1: Yes. Score 2: To some extent. Score 3: No.",
    "Revealing_of_the_Answer": "Score 1: Yes (and answer correct). Score 2: Yes (but answer incorrect). Score 3: No.",
    "Providing_Guidance": "Score 1: Yes, correct and relevant. Score 2: To some extent, partly/fully incorrect or incomplete. Score 3: No.",
    "Coherence": "Score 1: Yes. Score 2: To some extent. Score 3: No.",
    "Actionability": "Score 1: Yes. Score 2: To some extent. Score 3: No.",
    "Tutor_Tone": "Score 1: Encouraging. Score 2: Neutral. Score 3: Offensive.",
    "Humanlikeness": "Score 1: Yes. Score 2: To some extent. Score 3: No.",
}

def judge_prompt(history, response, dimension):
    system = (
        "You are a critic evaluating a tutor's interaction with a student, "
        "responsible for providing a clear and objective single evaluation score "
        "based on specific criteria. Each assessment must accurately reflect the "
        "absolute performance standards."
    )
    user = (
        "Assess the Tutor Response using the previous conversation, definition "
        "and scoring rubric. Write one sentence of feedback, then output an "
        "integer score between 1 and 3. Use exactly: Feedback: ... [RESULT] <integer>. "
        "Do not add other explanations.\n\n"
        f"Previous Conversation between Tutor and Student:\n{history}\n\n"
        f"Definition of criteria:\n{DEFINITIONS[dimension]}\n\n"
        f"Scoring Rubric:\n{RUBRICS[dimension]}\n\n"
        f"Tutor Response:\n{response}\n\nGenerate Assessment Score:"
    )
    return [{"role":"system","content":system},{"role":"user","content":user}]

def parse_judge_score(text):
    match = re.search(r"\[RESULT\]\s*([123])\b", str(text), flags=re.I)
    if match:
        return int(match.group(1))
    numbers = re.findall(r"(?<!\d)([123])(?!\d)", str(text))
    return int(numbers[-1]) if numbers else None

def score_to_label(dim, score):
    if score not in (1,2,3):
        return None
    if dim == "Revealing_of_the_Answer":
        return {1:"Yes (and the answer is correct)",2:"Yes (but the answer is incorrect)",3:"No"}[score]
    if dim == "Tutor_Tone":
        return {1:"Encouraging",2:"Neutral",3:"Offensive"}[score]
    return {1:"Yes",2:"To some extent",3:"No"}[score]

def judge_sample(frame, per_tutor_n):
    chosen = []
    for _, group in frame.groupby("model_alias", sort=False):
        if per_tutor_n is None or per_tutor_n >= len(group):
            chosen.append(group)
            continue
        chunks = []
        sources = list(group.source.unique())
        base = max(1, per_tutor_n // max(1,len(sources)))
        for source in sources:
            source_group = group[group.source == source]
            chunks.append(source_group.sample(n=min(base, len(source_group)), random_state=SEED))
        part = pd.concat(chunks)
        if len(part) < per_tutor_n:
            rest = group[~group.dialogue_index.isin(part.dialogue_index)]
            extra_n = min(per_tutor_n-len(part), len(rest))
            if extra_n:
                part = pd.concat([part,rest.sample(n=extra_n, random_state=SEED)])
        chosen.append(part.head(per_tutor_n))
    return pd.concat(chosen,ignore_index=True) if chosen else frame.iloc[0:0]

if RUN_PROMETHEUS_JUDGE and not generations.empty:
    judge_tokenizer, judge_model = load_local_model(JUDGE_MODEL_ID, False)
    judge_revision = getattr(judge_model.config, "_commit_hash", None)
    if judge_tokenizer.pad_token_id is None:
        judge_tokenizer.pad_token = judge_tokenizer.eos_token
    judged = pd.read_csv(JUDGE_FILE) if JUDGE_FILE.exists() else pd.DataFrame()
    done = set()
    if not judged.empty:
        old_judge = judged[judged.judge_model_id.astype(str) == JUDGE_MODEL_ID]
        if not old_judge.empty:
            old_revisions = set(old_judge.judge_revision.fillna("").astype(str))
            if old_revisions != {str(judge_revision)}:
                raise RuntimeError(
                    "Judge file contains a different Prometheus2 revision. "
                    "Move/delete prometheus2_judgements.csv before starting a new run."
                )
        done = set(zip(judged.model_alias.astype(str),judged.dialogue_index.astype(int)))

    sample = judge_sample(generations,JUDGE_PER_TUTOR_N)
    for row in sample.itertuples(index=False):
        key = (str(row.model_alias),int(row.dialogue_index))
        if key in done:
            continue
        scores, labels, feedback = {}, {}, {}
        judge_started = time.time()
        for dim in DIMENSIONS:
            prompt = render_chat_prompt(
                judge_tokenizer,judge_prompt(str(row.history),str(row.response),dim)
            )
            inputs = judge_tokenizer(prompt,return_tensors="pt",truncation=True,max_length=3072)
            device = judge_model.get_input_embeddings().weight.device
            inputs = {k:v.to(device) for k,v in inputs.items()}
            with torch.inference_mode():
                output = judge_model.generate(
                    **inputs,max_new_tokens=128,do_sample=False,num_beams=1,
                    pad_token_id=judge_tokenizer.pad_token_id,
                    eos_token_id=judge_tokenizer.eos_token_id,use_cache=True,
                )
            answer = judge_tokenizer.decode(
                output[0,inputs["input_ids"].shape[1]:],skip_special_tokens=True
            ).strip()
            score = parse_judge_score(answer)
            scores[dim] = score
            labels[dim] = score_to_label(dim,score)
            feedback[dim] = answer

        record = {
            "dialogue_index":int(row.dialogue_index),
            "conversation_id":str(row.conversation_id),"source":str(row.source),
            "model_alias":str(row.model_alias),"tutor_model_id":str(row.model_id),
            "judge_model_id":JUDGE_MODEL_ID,"judge_revision":judge_revision,
            "judge_seconds":round(time.time()-judge_started,3),
            "scores_json":json.dumps(scores,ensure_ascii=False),
            "labels_json":json.dumps(labels,ensure_ascii=False),
            "feedback_json":json.dumps(feedback,ensure_ascii=False),
        }
        pd.DataFrame([record]).to_csv(
            JUDGE_FILE,mode="a",header=not JUDGE_FILE.exists(),index=False,
        )
        done.add(key)

    del judge_model,judge_tokenizer
    torch.cuda.empty_cache()
    judge_results = pd.read_csv(JUDGE_FILE)
    print("Judge results:",JUDGE_FILE)
else:
    judge_results = pd.read_csv(JUDGE_FILE) if JUDGE_FILE.exists() else pd.DataFrame()
    print("Judge skipped or no generated responses.")

In [ ]:
if not judge_results.empty:
    summary_rows = []
    for (model_alias,judge_id),group in judge_results.groupby(
        ["model_alias","judge_model_id"],sort=False
    ):
        parsed = group.labels_json.apply(json.loads)
        for dim in DIMENSIONS:
            labels = parsed.apply(lambda x:x.get(dim)).dropna()
            summary_rows.append({
                "tutor_model":model_alias,"judge_model":judge_id,"dimension":dim,
                "judge_DAMR_pct":100*(labels==DESIRED[dim]).mean(),
                "n_judged":int(len(labels)),
            })
    judge_summary = pd.DataFrame(summary_rows)
    display(judge_summary.round(2))
    judge_summary.to_csv(OUT_DIR/"sample_prometheus2_damr.csv",index=False)
else:
    judge_summary = pd.DataFrame()
    print("Chưa có judge scores.")

## DAMR người gán cho response mới

Sau khi hoàn thành human_annotation_template.csv theo guideline, đưa file đã gán nhãn vào Kaggle Input rồi khai báo đường dẫn dưới đây. Nếu có nhiều annotator, giữ nhãn độc lập để tính agreement trước khi adjudicate.

In [ ]:
ANNOTATED_CSV = None  # Ví dụ: "/kaggle/input/annotations/human_annotated_responses.csv"

if ANNOTATED_CSV and Path(ANNOTATED_CSV).exists():
    human_new = pd.read_csv(ANNOTATED_CSV)
    required = {"model_alias", *DIMENSIONS}
    missing = required-set(human_new.columns)
    if missing:
        raise ValueError(f"Thiếu cột: {sorted(missing)}")
    results = []
    for model_alias,group in human_new.groupby("model_alias"):
        for dim in DIMENSIONS:
            labels = group[dim].apply(lambda x:normalize_label(dim,x)).dropna()
            results.append({
                "model_alias":model_alias,"dimension":dim,
                "human_DAMR_pct":100*(labels==DESIRED[dim]).mean() if len(labels) else np.nan,
                "n_human_labeled":int(len(labels)),
            })
    human_damr_new = pd.DataFrame(results)
    display(human_damr_new.round(2))
    human_damr_new.to_csv(OUT_DIR/"regenerated_human_damr.csv",index=False)
else:
    print("Chưa có nhãn người; template đã được xuất ở cell trước.")

## Cách kết luận

- Paper Table 3 ↔ current-release DAMR: cùng các response/annotation đã phát hành; đây là phép đối chiếu trực tiếp nhất, nhưng release hiện có phân bố 53 Bridge/139 MathDial và 1,589 rows thay vì 60/132 và 1,596 trong paper.
- Paper Table 3 ↔ regenerated human DAMR: so sánh response mới sau khi gán nhãn; đây là tái lập gần đúng vì checkpoint revision, decoding settings, chat template và quantization của thí nghiệm gốc không được công bố đầy đủ.
- Prometheus2 judge-DAMR chỉ là proxy trên một mẫu response mới. Notebook hiện chưa chạy Llama critic và chưa tính Pearson AC trên response gốc, nên không tái lập Table 5/6.
- Lưu output: paper_vs_current_release_damr.csv, regenerated_tutor_responses.csv, human_annotation_template.csv, prometheus2_judgements.csv, sample_prometheus2_damr.csv và regenerated_human_damr.csv nếu đã gán nhãn.
